In [ ]:
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [3]:
# ==========================================
# Milestone 1: Setup, Installation, and Model Loading
# ==========================================

# Step 1: Check GPU availability
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Step 2: Install Required Libraries
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth==2025.9.2
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2

# Step 3: Import Libraries and Set Random Seed
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
torch.manual_seed(42)  # Set for reproducibility

# Step 4: Define Model Parameters
max_seq_length = 256
dtype = None
load_in_4bit = True

# Step 5: Load the Pre-trained Llama 3.2 Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-1B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Step 6: Apply LoRA Adapters to the Model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Loaded unsloth/Llama-3.2-1B-Instruct with LoRA adapters on {device}")

Using device: cuda
  Using cached xformers-0.0.29.post3.tar.gz (8.5 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'xformers' when getting requirements to build wheel
==((====))==  Unsloth 2025.9.2: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.9.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Loaded unsloth/Llama-3.2-1B-Instruct with LoRA adapters on cuda


In [4]:
# ==========================================
# Milestone 2: Data Preparation & Formatting
# ==========================================

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Step 1: Load the Dataset into a Pandas dataframe
df = pd.read_csv("Lead Scoring.csv")

# Step 2: Choose relevant columns and clean data
text_columns = [
    'Lead Origin', 'Lead Source', 'Last Activity', 'Country', 'Specialization',
    'How did you hear about X Education', 'What is your current occupation',
    'What matters most to you in choosing a course', 'Tags'
]
df = df[text_columns + ['Converted']].dropna()
print(f"Dataset size after dropping NaNs: {len(df)} rows")

# Step 3: Split the Data into Train and Validation Sets
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Converted'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['Converted'])
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

# Step 4: Create Conversational Prompts for Each Lead
def create_conversations(dataframe):
    conversations = []
    for _, row in dataframe.iterrows():
        lead_info = "".join([f"{col}: {row[col]}\n" for col in text_columns])
        convo = [
            {"role": "user", "content": f"Analyze this lead information and classify if they will convert:\n\n{lead_info}"},
            {"role": "assistant", "content": "1" if row['Converted'] == 1 else "0"}
        ]
        conversations.append({"conversations": convo})
    return conversations

train_dataset = Dataset.from_list(create_conversations(train_df))
val_dataset = Dataset.from_list(create_conversations(val_df))

# Step 5: Apply Chat Template to format text (Without manual pre-tokenization)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.2")

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Milestone 2 completed successfully: Data prepared and formatted!")

Dataset size after dropping NaNs: 4534 rows
Train size: 3627, Validation size: 453


Map:   0%|          | 0/3627 [00:00<?, ? examples/s]

Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Milestone 2 completed successfully: Data prepared and formatted!


In [ ]:
!pip install --upgrade torchvision

In [5]:
# ==========================================
# Milestone 3: Model Training (SFTTrainer)
# ==========================================

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Step 1: Configure Training Arguments and SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False, # Can speed up training for short sequences
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        eval_strategy="steps",
        eval_steps=20,
        report_to="none",
    ),
)

# Step 2: Train the Model
trainer_stats = trainer.train()
print("Training completed successfully!")

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3627 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x79272c7d5e90>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7d6e80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7cbbf0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tblib/__init__.py", line 139, in as_traceback
    next_tb = sys.exc_info()[2].tb_next
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c7d6ed0>
Traceback (most recent call last):
  File "/usr/local/

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/453 [00:00<?, ? examples/s]

Exception ignored in: <_io.BytesIO object at 0x79272c7cbce0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c771940>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing while Python audit hooks run
BufferError: Existing exports of data: object cannot be re-sized
Exception ignored in: <_io.BytesIO object at 0x79272c76f470>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/filelock/_api.py", line 2221, in _audit_fork_safety
    def _audit_fork_safety(  # pragma: no cover - CPython disables tracing

Step,Training Loss,Validation Loss
20,0.355400,0.405772
40,0.358100,0.272182
60,0.245400,0.236617


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training completed successfully!
